# 09 · The Full Cascade

### Recap & why now
Notebook 08 keeps the drone upright. It does not know where the drone is, and it has no
opinion about where it should be.

The gap looks unbridgeable: the command is "be at $(2, 1, 2)$", the actuators want four
thrusts, and between them sit two coupled nonlinear equations. Writing one function from
position error to motor thrusts is genuinely hard — so nobody does. The standard answer
is four small easy problems stacked on top of each other.

### Learning objectives
1. Explain **why** control is cascaded, and what each layer converts into what.
2. Write position and velocity controllers producing a desired acceleration.
3. Convert that acceleration into a **thrust magnitude and a desired attitude**.
4. Close the loop and fly to a point in 3-D.
5. Tune the cascade, and show what breaks when an inner loop is slower than its outer one.

In [ ]:
# === Standard setup used throughout this notebook ========================
import numpy as np                 # NumPy = fast vector/matrix math, so we never hand-write loops for arithmetic.
import matplotlib.pyplot as plt     # Matplotlib is our plotting engine for every static figure below.
from matplotlib import animation   # Turns a list of frames into a playable movie (used for the animations).
from mpl_toolkits.mplot3d import Axes3D   # Registers the '3d' projection that every figure here needs.
from IPython.display import HTML    # Embeds an animation as a self-contained JS player (no ffmpeg required).

%matplotlib inline
# Render animations as an in-browser JavaScript player so they always play, on any machine.
plt.rcParams["animation.html"] = "jshtml"
# Raise the embed size cap (MB) so longer clips are not silently cut off.
plt.rcParams["animation.embed_limit"] = 60
# One consistent, readable look for every figure in the manual.
plt.rcParams.update({"figure.dpi": 80, "font.size": 11, "axes.grid": True})
# Print matrices with 3 decimals and no scientific notation, so output is easy to eyeball.
np.set_printoptions(precision=3, suppress=True)
print("Setup complete — NumPy", np.__version__, "| Matplotlib", plt.matplotlib.__version__)

In [ ]:
# === Orientation toolkit, built up over Notebooks 02-05 ==================

def quat_normalize(q):
    """Force |q| = 1. Integration drifts off the unit sphere; this pulls it back."""
    q = np.asarray(q, float)
    return q/np.linalg.norm(q)

def quat_multiply(a, b):
    """Hamilton product a (x) b: 'do b first, then a', the same reading order as matrices."""
    aw, ax, ay, az = a
    bw, bx, by, bz = b
    return np.array([aw*bw - ax*bx - ay*by - az*bz,     # Scalar part.
                     aw*bx + ax*bw + ay*bz - az*by,     # Vector part, x.
                     aw*by - ax*bz + ay*bw + az*bx,     #              y.
                     aw*bz + ax*by - ay*bx + az*bw])    #              z.

def quat_conjugate(q):
    """Flip the vector part — for a unit quaternion this is the INVERSE rotation."""
    return np.array([q[0], -q[1], -q[2], -q[3]])

def quat_to_rotmat(q):
    """The body-to-world rotation matrix that this quaternion represents."""
    w, x, y, z = quat_normalize(q)
    return np.array([[1-2*(y*y+z*z),   2*(x*y-w*z),   2*(x*z+w*y)],
                     [  2*(x*y+w*z), 1-2*(x*x+z*z),   2*(y*z-w*x)],
                     [  2*(x*z-w*y),   2*(y*z+w*x), 1-2*(x*x+y*y)]])

def euler_to_quat(roll, pitch, yaw):
    """ZYX Euler angles -> quaternion. Used to SET a pose, never to store one."""
    cr, sr = np.cos(roll/2), np.sin(roll/2)
    cp, sp = np.cos(pitch/2), np.sin(pitch/2)
    cy, sy = np.cos(yaw/2), np.sin(yaw/2)
    return np.array([cr*cp*cy + sr*sp*sy, sr*cp*cy - cr*sp*sy,
                     cr*sp*cy + sr*cp*sy, cr*cp*sy - sr*sp*cy])

def quat_to_euler(q):
    """Quaternion -> roll, pitch, yaw. For DISPLAY only — never as simulator state."""
    w, x, y, z = quat_normalize(q)
    return np.array([np.arctan2(2*(w*x + y*z), 1 - 2*(x*x + y*y)),
                     np.arcsin(np.clip(2*(w*y - z*x), -1, 1)),      # clip guards against 1+1e-16.
                     np.arctan2(2*(w*z + x*y), 1 - 2*(y*y + z*z))])

def quat_from_rotmat(R):
    """Rotation matrix -> quaternion. Four branches, so we never divide by a small number."""
    tr = np.trace(R)
    if tr > 0:
        s_ = np.sqrt(tr + 1.0)*2
        q = np.array([0.25*s_, (R[2,1]-R[1,2])/s_, (R[0,2]-R[2,0])/s_, (R[1,0]-R[0,1])/s_])
    elif R[0,0] > R[1,1] and R[0,0] > R[2,2]:
        s_ = np.sqrt(1.0 + R[0,0] - R[1,1] - R[2,2])*2
        q = np.array([(R[2,1]-R[1,2])/s_, 0.25*s_, (R[0,1]+R[1,0])/s_, (R[0,2]+R[2,0])/s_])
    elif R[1,1] > R[2,2]:
        s_ = np.sqrt(1.0 + R[1,1] - R[0,0] - R[2,2])*2
        q = np.array([(R[0,2]-R[2,0])/s_, (R[0,1]+R[1,0])/s_, 0.25*s_, (R[1,2]+R[2,1])/s_])
    else:
        s_ = np.sqrt(1.0 + R[2,2] - R[0,0] - R[1,1])*2
        q = np.array([(R[1,0]-R[0,1])/s_, (R[0,2]+R[2,0])/s_, (R[1,2]+R[2,1])/s_, 0.25*s_])
    return quat_normalize(q)

def axis_angle_to_quat(axis, angle):
    """Build a quaternion from 'rotate by `angle` about `axis`' — the geometric reading."""
    axis = np.asarray(axis, float); axis = axis/np.linalg.norm(axis)
    return np.array([np.cos(angle/2), *(axis*np.sin(angle/2))])

def quat_rotate(q, v):
    """Rotate v from the body frame into the world frame, using the sandwich product."""
    return quat_multiply(quat_multiply(q, np.array([0.0, *v])), quat_conjugate(q))[1:]

# === The vehicle, and how to draw it =====================================

PARAMS = dict(m=1.0, L=0.25,                       # Mass [kg] and hub-to-rotor distance [m].
              I=np.diag([0.01, 0.01, 0.02]),       # Inertia [kg m^2]; yaw is the heavy axis.
              d=0.016,                             # Drag torque per newton of thrust [m].
              T_min=0.0, T_max=6.0)                # What one motor can produce [N].
g = 9.81                                           # Gravity [m/s^2], along world -z.

ARM = PARAMS["L"]/np.sqrt(2)                       # Each rotor sits ARM along body x AND body y.
MOTOR_POS = np.array([[ ARM, -ARM, 0.0],           # M1 front-right.
                      [ ARM,  ARM, 0.0],           # M2 front-left.
                      [-ARM,  ARM, 0.0],           # M3 rear-left.
                      [-ARM, -ARM, 0.0]])          # M4 rear-right.
SPIN = np.array([-1.0, 1.0, -1.0, 1.0])            # +1 = counter-clockwise seen from above.

def draw_quad(ax, position, q, scale=3.0, thrusts=None):
    """Draw the drone: four arms, four rotors, a nose marker and the thrust arrow."""
    R = quat_to_rotmat(q)                          # Body-to-world, so body points become world points.
    for i, mp in enumerate(MOTOR_POS):
        tip = np.asarray(position, float) + R @ (mp*scale)
        seg = np.array([position, tip])
        ax.plot(seg[:, 0], seg[:, 1], seg[:, 2], color="0.35", lw=2)
        shade = "C3" if i in (0, 1) else "C0"      # Front rotors red, rear blue, so the nose is visible.
        if thrusts is not None:
            load = np.clip(thrusts[i]/PARAMS["T_max"], 0, 1)
            shade = plt.cm.YlOrRd(0.3 + 0.7*load)  # Colour by how hard the motor is working.
        ax.plot([tip[0]], [tip[1]], [tip[2]], "o", ms=6, color=shade)
    ax.quiver(*position, *(R[:, 2]*0.9), color="C1", lw=2.2, arrow_length_ratio=0.25)

def set_3d(ax, xlim, ylim, zlim):
    """Equal-ish 3-D axes with explicit limits, so animations do not jitter."""
    ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_zlim(*zlim)
    ax.set_box_aspect([xlim[1]-xlim[0], ylim[1]-ylim[0], zlim[1]-zlim[0]])
    ax.set_xlabel("x — East [m]"); ax.set_ylabel("y — North [m]"); ax.set_zlabel("z — Up [m]")

print("vehicle ready: %.1f kg, hover %.2f N total, %.3f N per motor, thrust/weight %.2f" %
      (PARAMS["m"], PARAMS["m"]*g, PARAMS["m"]*g/4, 4*PARAMS["T_max"]/(PARAMS["m"]*g)))

# === Mixing and 6-DOF dynamics, from Notebooks 06-07 =====================

POS, VEL, QUAT, OMEGA = slice(0, 3), slice(3, 6), slice(6, 10), slice(10, 13)
MIX = np.vstack([np.ones(4), MOTOR_POS[:, 1], -MOTOR_POS[:, 0], -SPIN*PARAMS["d"]])

def motor_mixer(total_thrust, torques, p=PARAMS):
    """Desired wrench -> four motor thrusts, clipped to what the hardware can do."""
    T4 = np.linalg.solve(MIX, np.array([total_thrust, *torques], float))
    return np.clip(T4, p["T_min"], p["T_max"])     # A propeller cannot pull, nor push forever.

def quad_dynamics(state, motor_thrusts, p=PARAMS, f_ext=np.zeros(3)):
    """x_dot for the 13-state quadcopter, driven by four motor thrusts."""
    q = quat_normalize(state[QUAT]); w = state[OMEGA]
    T, tx, ty, tz = MIX @ np.asarray(motor_thrusts, float)          # Geometry does its job here.
    v_dot = (quat_to_rotmat(q) @ np.array([0.0, 0.0, T])            # Thrust, body -> world.
             + np.array([0.0, 0.0, -p["m"]*g]) + f_ext)/p["m"]      # Gravity, ENU, plus any push.
    q_dot = 0.5*quat_multiply(q, np.array([0.0, *w]))               # Notebook 05's kinematics.
    w_dot = np.linalg.solve(p["I"], np.array([tx, ty, tz]) - np.cross(w, p["I"] @ w))
    return np.concatenate([state[VEL], v_dot, q_dot, w_dot])

def rk4_step(state, motor_thrusts, dt, p=PARAMS, f_ext=np.zeros(3)):
    """One RK4 step, followed by the renormalisation Notebook 05 insisted on."""
    k1 = quad_dynamics(state, motor_thrusts, p, f_ext)
    k2 = quad_dynamics(state + 0.5*dt*k1, motor_thrusts, p, f_ext)
    k3 = quad_dynamics(state + 0.5*dt*k2, motor_thrusts, p, f_ext)
    k4 = quad_dynamics(state + dt*k3, motor_thrusts, p, f_ext)
    s = state + dt/6*(k1 + 2*k2 + 2*k3 + k4)
    s[QUAT] = quat_normalize(s[QUAT])
    return s

def make_state(p=(0, 0, 0), v=(0, 0, 0), q=(1, 0, 0, 0), w=(0, 0, 0)):
    """Assemble the 13-element state vector."""
    return np.concatenate([p, v, q, w]).astype(float)

def simulate(command, T_end=4.0, dt=0.005, s0=None, p=PARAMS, f_ext=lambda t: np.zeros(3)):
    """Fly the drone. `command(t, state)` returns four motor thrusts in newtons."""
    s = make_state() if s0 is None else np.array(s0, float)
    ts, xs, ms = [0.0], [s.copy()], []
    for k in range(int(round(T_end/dt))):
        T4 = np.clip(np.asarray(command(k*dt, s), float), p["T_min"], p["T_max"])
        s = rk4_step(s, T4, dt, p, f_ext(k*dt))
        ts.append((k+1)*dt); xs.append(s.copy()); ms.append(T4)
    return np.array(ts), np.array(xs), np.array(ms)

T_HOVER = PARAMS["m"]*g                            # Total thrust that exactly cancels weight.
HOVER_EACH = T_HOVER/4                             # ...split over four identical motors.
print("model ready — hover needs %.4f N total, %.4f N per motor" % (T_HOVER, HOVER_EACH))

## 1 · The stack

```text
   "go to (2, 1, 2)"        position error       SLOW
            ▼   POSITION CONTROLLER
      desired velocity
            ▼   VELOCITY CONTROLLER
    desired acceleration
            ▼   THRUST + ATTITUDE   ← the pivot: linear becomes rotational
   thrust magnitude + desired attitude
            ▼   ATTITUDE CONTROLLER          (Notebook 08)
    desired angular rate
            ▼   RATE CONTROLLER              FAST
      desired torque
            ▼   MOTOR MIXER                  (Notebook 07)
       four motor thrusts
```

Each box is a proportional controller on a three-vector — a line or two of code. The
difficulty was never in any one box; it was in the coupling, and ordering the boxes
dissolves it.

The pivot in the middle is forced on us by physics: a quadcopter cannot produce sideways
force, so **the only way to ask for horizontal acceleration is to ask for a tilt**.

In [ ]:
# === The cascade, built in Notebooks 08-09 ===============================

GAINS = dict(Kp=np.array([2.0, 2.0, 3.0]),         # Position -> velocity; z is stiffer.
             Kv=np.array([4.0, 4.0, 5.0]),         # Velocity -> acceleration.
             K_R=np.array([12.0, 12.0, 6.0]),      # Attitude -> angular rate; yaw softer.
             K_w=np.array([0.06, 0.06, 0.06]),     # Angular rate -> torque.
             v_max=4.0,                            # Speed clamp [m/s].
             tilt_max=np.deg2rad(35))              # How far the COMMAND may ask the drone to lean.

def acc_to_thrust_attitude(a_cmd, yaw_des, q, p=PARAMS, K=GAINS):
    """The pivot: desired acceleration -> (thrust magnitude, desired attitude)."""
    F = p["m"]*(a_cmd + np.array([0.0, 0.0, g]))   # ENU: +g compensates gravity.
    fz = max(F[2], 0.4*p["m"]*g)                   # Never let the vertical part collapse.
    fxy = F[:2]; max_xy = fz*np.tan(K["tilt_max"])
    if np.linalg.norm(fxy) > max_xy:
        fxy = fxy*max_xy/np.linalg.norm(fxy)       # Cap the tilt the command may request.
    F = np.array([fxy[0], fxy[1], fz])
    T = float(F @ quat_to_rotmat(q)[:, 2])         # Project onto the axis we can actually push along.
    z_des = F/np.linalg.norm(F)
    x_c = np.array([np.cos(yaw_des), np.sin(yaw_des), 0.0])
    y_des = np.cross(z_des, x_c); y_des /= np.linalg.norm(y_des)
    return T, quat_from_rotmat(np.column_stack([np.cross(y_des, z_des), y_des, z_des]))

def attitude_controller(q, q_des, K=GAINS):
    """Orientation error -> desired body angular rate."""
    q_e = quat_multiply(quat_conjugate(q), q_des)  # Rotation from current TO desired, in body axes.
    if q_e[0] < 0:
        q_e = -q_e                                 # Short way round — Notebook 04, Section 4.
    return 2.0*K["K_R"]*q_e[1:]

def rate_controller(w_cmd, w, p=PARAMS, K=GAINS):
    """Angular-rate error -> body torque, with the gyroscopic term fed forward."""
    return K["K_w"]*(w_cmd - w) + np.cross(w, p["I"] @ w)

def cascaded_controller(state, p_des, v_ff, a_ff, yaw_des, p=PARAMS, K=GAINS):
    """One pass through the whole stack: state + reference -> four motor thrusts."""
    v_cmd = K["Kp"]*(p_des - state[POS]) + v_ff
    speed = np.linalg.norm(v_cmd)
    if speed > K["v_max"]:
        v_cmd = v_cmd*K["v_max"]/speed             # Keep the direction, cap the magnitude.
    a_cmd = K["Kv"]*(v_cmd - state[VEL]) + a_ff
    T, q_des = acc_to_thrust_attitude(a_cmd, yaw_des, state[QUAT], p, K)
    w_cmd = attitude_controller(state[QUAT], q_des, K)
    tau = rate_controller(w_cmd, state[OMEGA], p, K)
    return motor_mixer(T, tau, p)

def fly(reference, T_end, dt=0.005, s0=None, p=PARAMS, K=GAINS,
        yaw_of_t=lambda t: 0.0, f_ext=lambda t: np.zeros(3)):
    """Closed-loop flight. `reference(t)` returns (p_des, v_ff, a_ff)."""
    s = make_state() if s0 is None else np.array(s0, float)
    ts, xs, ms, rs = [0.0], [s.copy()], [], []
    for k in range(int(round(T_end/dt))):
        p_des, v_ff, a_ff = reference(k*dt)
        T4 = cascaded_controller(s, p_des, v_ff, a_ff, yaw_of_t(k*dt), p, K)
        s = rk4_step(s, T4, dt, p, f_ext(k*dt))
        ts.append((k+1)*dt); xs.append(s.copy()); ms.append(T4); rs.append(p_des)
    return np.array(ts), np.array(xs), np.array(ms), np.array(rs)

hold = lambda target: (lambda t: (np.array(target, float), np.zeros(3), np.zeros(3)))
print("cascade loaded — six small controllers, each feeding the next")

## 2 · From acceleration to a tilt

$$F_{\text{des}} = m\,(a_{\text{cmd}} + [0, 0, g])$$

That $+g$ is gravity compensation, and in **ENU** it is a plus: gravity pulls along
$-z$, so simply holding still requires pushing $+mg$ along $+z$. Ask for zero
acceleration and you should get exactly hover thrust — which is the check that this
sign is right.

Then split $F_{\text{des}}$ into a magnitude and a direction. The magnitude is its
**projection onto the current body $z$**, not its length: while the drone is still
rotating into place, it should ask only for the part it can actually deliver.

In [ ]:
q_level = np.array([1.0, 0.0, 0.0, 0.0])
print("  requested acceleration      thrust [N]   desired roll/pitch/yaw [deg]")
for a_cmd in [np.zeros(3), np.array([0, 0, 2.0]), np.array([2.0, 0, 0]),
              np.array([0, 3.0, 0]), np.array([12.0, 0, 0])]:
    T, q_des = acc_to_thrust_attitude(a_cmd, 0.0, q_level)
    print("  %-26s %10.3f   %s" % (np.round(a_cmd, 1), T, np.round(np.degrees(quat_to_euler(q_des)), 2)))

print("\nRow 1: zero acceleration asks for exactly %.2f N — hover. The +g sign is right." % T_HOVER)
print("Row 3: 2 m/s^2 East asks for a nose-down PITCH, because pitching tips thrust toward +x.")
print("Row 4: 3 m/s^2 North asks for ROLL instead.")
print("Row 5: an absurd 12 m/s^2 is clipped to the %.0f° tilt limit." % np.degrees(GAINS["tilt_max"]))

## 3 · Flying to a point

The whole cascade is six function calls, every one of them a gain times an error. Below
it takes off to 2 m, then flies a diagonal to $(2, 1, 2)$ — the manoeuvre that exercises
the pivot, because the drone must tilt, translate, and then *un*-tilt at exactly the
right moment.

In [ ]:
t, X, M, R = fly(hold([0, 0, 2.0]), T_end=6.0)
z = X[:, 2]
settle = t[next(i for i in range(len(z)) if np.all(np.abs(z[i:] - 2.0) < 0.05))]
print("climb to 2 m: final %.4f m, overshoot %.3f m, settled in %.2f s, peak motor %.2f N" %
      (z[-1], max(0, z.max()-2.0), settle, M.max()))

t2, X2, M2, R2 = fly(hold([2.0, 1.0, 2.0]), T_end=10.0)
rpy2 = np.degrees(np.array([quat_to_euler(q_) for q_ in X2[:, QUAT]]))
err2 = np.linalg.norm(X2[:, POS] - np.array([2.0, 1.0, 2.0]), axis=1)
print("\nmove to (2, 1, 2): final error %.4f m, within 10 cm after %.2f s" %
      (err2[-1], t2[np.argmax(err2 < 0.1)]))
print("peak tilt %.1f° (the command was limited to %.0f°), motors saturated %.1f%% of the flight" %
      (np.abs(rpy2[:, :2]).max(), np.degrees(GAINS["tilt_max"]),
       100*np.mean(M2.max(axis=1) >= PARAMS["T_max"]-1e-9)))

fig, axes = plt.subplots(1, 3, figsize=(14, 3.2))
for i, lbl in enumerate(["x", "y", "z"]):
    axes[0].plot(t2, X2[:, i], lw=1.7, label=lbl)
    axes[0].axhline([2.0, 1.0, 2.0][i], color="0.7", ls="--", lw=1)
axes[0].set_ylabel("position [m]"); axes[0].legend(fontsize=8); axes[0].set_title("Getting there")
for i, lbl in enumerate(["roll", "pitch"]):
    axes[1].plot(t2, rpy2[:, i], lw=1.7, label=lbl)
axes[1].axhline(np.degrees(GAINS["tilt_max"]), color="C3", ls=":", lw=1.2)
axes[1].set_ylabel("angle [deg]"); axes[1].legend(fontsize=8); axes[1].set_title("Tilt: out, then back")
for i in range(4):
    axes[2].plot(t2[1:], M2[:, i], lw=1.1, label="M%d" % (i+1))
axes[2].axhline(PARAMS["T_max"], color="C3", ls="--", lw=1.1)
axes[2].set_ylabel("motor thrust [N]"); axes[2].legend(fontsize=7, ncol=2); axes[2].set_title("Motors")
for a_ in axes: a_.set_xlabel("time [s]")
plt.tight_layout(); plt.show()

## 4 · Two honest observations

**The motors saturate at the start.** A step command asks for maximum effort
immediately and the mixer clips. Nothing breaks — the drone gets less than it asked for
and carries on — but during those moments the limits are in charge, not the controller.
Notebook 10 removes this by never issuing a step in the first place.

**The peak tilt slightly exceeds the 35° limit.** That is not a bug in the limiter:
`acc_to_thrust_attitude` caps what the command may *request*, while the attitude loop
has its own momentum and overshoots the request. A saturation applied to a command is
not a guarantee about the response.

In [ ]:
def variant(**changes):
    K = {k_: (v.copy() if isinstance(v, np.ndarray) else v) for k_, v in GAINS.items()}
    K.update(changes); return K

target = [2.0, 1.0, 2.0]
cases = [("baseline               ", GAINS),
         ("Kp x 4 (twitchy outer) ", variant(Kp=np.array([8.0, 8.0, 3.0]))),
         ("Kv / 2 (weak damping)  ", variant(Kv=np.array([2.0, 2.0, 5.0]))),
         ("K_R / 4 (slow attitude)", variant(K_R=np.array([3.0, 3.0, 6.0]))),
         ("K_w / 4 (slow rate)    ", variant(K_w=np.array([0.015, 0.015, 0.06])))]

print("  gains                    to 10 cm    settle    final error   peak tilt")
for name, K in cases:
    tt, XX, MM, _ = fly(hold(target), T_end=12.0, K=K)
    e = np.linalg.norm(XX[:, POS] - np.array(target), axis=1)
    rp = np.degrees(np.array([quat_to_euler(q_) for q_ in XX[:, QUAT]]))
    inside = [i for i in range(len(e)) if np.all(e[i:] < 0.05)]
    print("  %s %10s %9s %13.3f %10.1f°" %
          (name, "%.2f s" % tt[np.argmax(e < 0.1)] if np.any(e < 0.1) else " > 12",
           "%.2f s" % tt[inside[0]] if inside else " > 12", e[-1], np.abs(rp[:, :2]).max()))

print("\nWeakening the OUTER gains is disappointing but safe — the drone is slower or untidier")
print("and still arrives. Weakening the INNER ones is a different kind of failure: the velocity")
print("loop is now commanding attitude changes faster than the attitude loop can deliver them,")
print("so the drone chases a setpoint it never reaches. That is the cascade's founding assumption")
print("breaking — each layer can only treat the layer below as 'already done' if it is faster.")

## 🧪 Try it yourself

**E1.** `acc_to_thrust_attitude` projects $F_{\text{des}}$ onto the current body $z$
instead of using its length. What would go wrong at the start of a large horizontal
move if we used the length?

**E2.** Fly to $(3, 0, 2)$ while commanding a 90° yaw. Does the yaw disturb the position
tracking? Compare against the same flight with no yaw command.

In [ ]:
# --- Solution E1 ---
q_lvl = np.array([1.0, 0, 0, 0])
F = PARAMS["m"]*(np.array([8.0, 0, 0]) + np.array([0, 0, g]))
print("E1: for a_cmd = [8, 0, 0], |F_des| = %.2f N but its projection on body z is %.2f N." %
      (np.linalg.norm(F), F @ quat_to_rotmat(q_lvl)[:, 2]))
print("    Using the length would command that full thrust STRAIGHT UP while the drone is still")
print("    level — it would balloon upward before it had rotated at all. The projection asks only")
print("    for the component it can currently deliver, and grows naturally as the drone tilts into")
print("    place. It is the difference between 'what I want' and 'what I can do about it yet'.")

# --- Solution E2 ---
yaw_ramp = lambda t_: np.deg2rad(90)*min(t_/3.0, 1.0)
t_y, X_y, M_y, R_y = fly(hold([3.0, 0, 2.0]), T_end=10.0, yaw_of_t=yaw_ramp)
t_n, X_n, M_n, R_n = fly(hold([3.0, 0, 2.0]), T_end=10.0)
e_y = np.linalg.norm(X_y[:, POS] - np.array([3., 0, 2.]), axis=1)
e_n = np.linalg.norm(X_n[:, POS] - np.array([3., 0, 2.]), axis=1)
rpy_y = np.degrees(np.array([quat_to_euler(q_) for q_ in X_y[:, QUAT]]))

print("\nE2: final error with a 90° yaw command %.4f m, without it %.4f m" % (e_y[-1], e_n[-1]))
print("    largest difference between the two position traces: %.4f m" % np.abs(e_y - e_n).max())
print("    final yaw %.2f° (commanded 90°)" % rpy_y[-1, 2])
print("    Yaw and position are almost perfectly decoupled, as they should be: the desired")
print("    attitude construction uses the ACCELERATION to pick the tilt, and the yaw command only")
print("    to decide where the nose sits within that tilt. On a real drone the coupling is not")
print("    quite zero — rotor drag, frame asymmetry — but it is small enough to treat as zero.")

## 🚁 Mini-project: a waypoint mission

Take off, cross the room, turn to face a new heading, and settle. No hand-fitted
schedule, no per-manoeuvre tuning — just a list of waypoints and the same six functions.
Change the mass or the starting point and it still works, which is the entire difference
between this notebook and Notebook 07.

In [ ]:
waypoints = [(0, 0, 1.5), (3.0, 2.0, 1.5), (3.0, 2.0, 1.5)]
hold_times = [4.0, 5.0, 4.0]
edges = np.cumsum(hold_times)

def mission(t_):
    """Hold each waypoint for its allotted time, then step to the next."""
    i = int(np.searchsorted(edges, t_, side="right"))
    return np.array(waypoints[min(i, len(waypoints)-1)], float), np.zeros(3), np.zeros(3)

yaw_cmd = lambda t_: np.deg2rad(120)*np.clip((t_ - 9.0)/2.5, 0, 1)
t, X, M, R = fly(mission, T_end=float(edges[-1]), yaw_of_t=yaw_cmd)
rpy = np.degrees(np.array([quat_to_euler(q_) for q_ in X[:, QUAT]]))
print("arrived at %s, commanded %s" % (np.round(X[-1, POS], 3), np.round(R[-1], 2)))
print("final yaw %.1f°, peak tilt %.1f°, motors saturated %.1f%% of the flight" %
      (rpy[-1, 2], np.abs(rpy[:, :2]).max(), 100*np.mean(M.max(axis=1) >= PARAMS["T_max"]-1e-9)))

step = 14
fig = plt.figure(figsize=(6.8, 5.4))
ax = fig.add_subplot(111, projection="3d")

def frame(j):
    ax.clear()
    k = step*j
    ax.plot(X[:k+1, 0], X[:k+1, 1], X[:k+1, 2], color="C0", lw=1.8)
    for wp in waypoints[:2]:
        ax.plot([wp[0]], [wp[1]], [wp[2]], "*", color="C2", ms=13)
    draw_quad(ax, X[k, POS], X[k, QUAT], scale=2.2, thrusts=M[min(k, len(M)-1)])
    set_3d(ax, (-1, 4.5), (-1, 3.5), (0, 3))
    ax.set_title("t = %4.1f s   pos %s   yaw %5.1f°" % (k*0.005, np.round(X[k, POS], 2), rpy[k, 2]),
                 fontsize=9)
    ax.view_init(elev=24, azim=-62)
    return []

anim = animation.FuncAnimation(fig, frame, frames=len(X)//step, interval=50, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

> **🤖 Robotics connection.** This cascade is what runs on the drone in your nearest
> camera shop. PX4 and ArduPilot implement exactly these layers under exactly these
> names; the differences are refinements rather than restructuring — integral terms for
> steady wind, feedforward from the trajectory, saturation handling that protects
> attitude over thrust, and a rate loop at kilohertz instead of our 200 Hz. The reason
> the architecture has outlived so many control fashions is that it fails gracefully: if
> the position layer produces nonsense, the attitude layer still keeps the vehicle
> upright while you work out why.

**Where next.** Look at the motor plot once more. The step command asked for everything
at once and the mixer clipped. Notebook 10 puts a **trajectory** in front of the cascade
and the difference is larger than you would guess.